# E-Commerce Lakehouse: Bronze Order Items

## Load the source order items

Read the original Olist order items CSV from the source Volume.
Preserve source values as strings before applying Silver transformations.

In [0]:
raw_order_items_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/ecommerce/raw_files/olist_order_items_dataset.csv")
)

display(raw_order_items_df.limit(5))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


## Inspect the source structure and row count

Count the order item records and inspect their data types.
Establish a baseline before saving the Bronze table.

In [0]:
print("Source order item rows:", raw_order_items_df.count())
raw_order_items_df.printSchema()

Source order item rows: 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



## Add ingestion metadata

Add ingestion_timestamp to record when the order item data is
processed. Preserve all original source columns and values.

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_order_items_df = raw_order_items_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

display(bronze_order_items_df.limit(5))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,2026-10-17T20:17:48.710Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,2026-10-17T20:17:48.710Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,2026-10-17T20:17:48.710Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,2026-10-17T20:17:48.710Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,2026-10-17T20:17:48.710Z


## Refresh and validate Bronze order items

Rebuild the Bronze table from the complete source dataset.
Check for 112,650 source rows before replacing existing contents,
then verify the saved row count.

In [0]:
table_name = "workspace.ecommerce.bronze_order_items"

source_count = raw_order_items_df.count()

if source_count != 112650:
    raise ValueError(
        f"Expected 112,650 source items; found {source_count}. "
        "Stopped before replacing the table."
    )

(
    bronze_order_items_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_bronze_order_items_df = spark.table(table_name)
saved_count = saved_bronze_order_items_df.count()

if saved_count != source_count:
    raise ValueError("Saved Bronze item count does not match the source.")

print("Bronze order items refreshed:", saved_count)

Bronze order items refreshed: 112650
